# Module 1: Agent Loop + Tools

~3 min setup + exploration

Build a Dussault agent with tools that query the 2004 New England Patriots
datasets. Each tool extracts one structured data dimension — the same architectural
pattern NFL Next Gen Stats uses (feature extraction → inference → human-guided output).

By the end of this module you'll have an agent that can look up any player on the
97-person roster, pull game-by-game results across all 19 games (16 regular + 3 playoff),
and retrieve detailed stat lines for 9 key contributors.

**Prerequisites:** Python 3.10+, AWS credentials configured with Bedrock access (us-west-2)

In [ ]:
# Install dependencies
%pip install -q strands-agents strands-agents-tools

import sys
sys.path.insert(0, "../shared")

---

## The Agent Loop

A strands agent runs a loop:

1. Receive user query
2. Decide which tool(s) to call (or respond directly)
3. Execute the tool, feed the result back to the model
4. Repeat until the model has enough data to respond

Each tool is a **feature extractor** — it surfaces one structured data dimension.
The model is the **inference engine** that combines features into a narrative.

| NGS pattern | strands equivalent | what it does here |
| --- | --- | --- |
| 11 features per play | 5 tools | roster, position, game result, stats, coaching staff |
| trained model | Nova Pro | combines structured data into analysis |
| expert-approved output | system prompt | enforces evidence-first, narrative-aware voice |

The agent never guesses stats — it calls a tool first. That's the contract.

In [ ]:
from strands import Agent
from model_provider import get_model
from dussault_tools import (
    lookup_player,
    get_roster_by_position,
    get_game_result,
    get_season_stats,
    get_coaching_staff,
)

SYSTEM_PROMPT = """You are Dussault, a 2004 New England Patriots Dussault. Your approach mirrors
the best of patriots.com's coverage — deeply researched, evidence-first, narrative-aware.

When answering:
- Always look up the data before making claims. Never guess stats.
- Connect facts to story — why something happened matters as much as what happened.
- If a question is ambiguous (which game? which player?), ask for clarity.
- If the data isn't in your tools, say so clearly rather than fabricating.
- Be specific: cite game weeks, scores, stat lines, not vague superlatives.
- Describe players in terms of their role on the team, not isolated glory.
"""

agent = Agent(
    model=get_model(),
    tools=[lookup_player, get_roster_by_position, get_game_result, get_season_stats, get_coaching_staff],
    system_prompt=SYSTEM_PROMPT,
    callback_handler=None,
)

print("Dussault ready. 5 tools loaded.")

---

## Try It — Query the Roster

The agent has to look up actual data before answering. Watch the tool calls —
it will use `get_roster_by_position` or `lookup_player` to gather evidence
before composing a response.

In [ ]:
result = agent("Who were the Pro Bowlers on the 2004 Patriots?")
print(result)

---

## Game Results

The `get_game_result` tool takes a week number and returns the score, opponent,
and key performers. The model figures out which week(s) to query based on the
natural language input.

In [ ]:
result = agent("What happened in the AFC Championship game?")
print(result)

---

## Player Stats

Detailed season stats are available for 9 key contributors: Brady, Dillon,
Branch, Givens, Seymour, Harrison, Vinatieri, Bruschi, and McGinest.
The agent combines roster data with stat lines to tell the player's story
in the context of the team.

In [ ]:
result = agent("Tell me about Corey Dillon's 2004 season.")
print(result)

---

## Try It Yourself

Some questions to explore:

- "Who was the defensive coordinator and what was his scheme?"
- "How did the Patriots do in their three playoff games?"
- "Compare Brady's stats to Dillon's — who carried the offense?"
- "What was the win streak and how did it end?"
- "Tell me about the secondary — who played corner?"

In [ ]:
# Try your own query
# result = agent("your question here")
# print(result)

---

## What's Next

The agent loop works: query → tool call → inference → response. But we have
no visibility into what's happening between the user and the model.

**Module 2** adds **hooks** — lightweight observers that fire on every tool call.
You'll track query patterns, detect repeated lookups, and enforce a
cite-your-source guardrail without changing the agent's core logic.